# Table S1 — the primary result and its prespecified sensitivity

**The estimate and its principal design caveat, together.** The common odds ratio and
the six cumulative risk differences, then `[§13]`'s full-covariate propensity arm in a separated
block. The arm is prespecified and it is what sizes the residual imbalance `[§9 amendment]`, so
burying it in a supplement would leave the reader with an estimate and no measure of the one problem
the design has.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The two specifications

They differ in the propensity model and in nothing else: same estimator, same estimand, same
population — a claim the record computes rather than asserts.

In [ ]:
def block(label, specification, ps, balance_, estimate, intervals, tested):
    """Design rows then effect rows, for one propensity specification."""
    worst, _ = balance_.worst()
    beta = intervals["beta"]
    design = [
        ("patients in the propensity model", report.fmt(int(ps.in_model.sum())), *("—",) * 4),
        *((f"effective sample size, {ARM[a]}", report.fmt(ps.ess[a]), *("—",) * 4) for a in ARMS),
        (f"worst residual SMD ({worst.covariate})", report.fmt(worst.weighted), *("—",) * 4),
    ]
    # The interval note promises a surviving draw count beside each interval [§10], so it is a column.
    effect = [
        ("common odds ratio exp(beta)", report.fmt(estimate.odds_ratio),
         report.fmt(np.exp(beta.lo)), report.fmt(np.exp(beta.hi)),
         report.fmt(beta.p) if tested else "—", str(beta.n_draws)),
        *((f"RD_{k}  P(mRS<=k) difference", report.fmt(estimate.rd[k]),
           report.fmt(intervals[f"rd_{k}"].lo), report.fmt(intervals[f"rd_{k}"].hi), "—",
           str(intervals[f"rd_{k}"].n_draws))
          for k in C.MRS_THRESHOLDS),
    ]
    return [(label, specification, kind, *row)
            for kind, rows in (("design", design), ("effect", effect)) for row in rows]


table = pd.DataFrame(
    block("primary [§7]", bundle.ps.spec.label, bundle.ps, bundle.balance, bundle.primary,
          bundle.boot.intervals, tested=True)
    + block("full-covariate arm [§13]", bundle.arm.ps.spec.label, bundle.arm.ps, bundle.arm.balance,
            bundle.arm.estimate, bundle.arm.intervals, tested=False),
    columns=["row", "specification", "block", "quantity", "estimate", "ci lo", "ci hi", "p",
             "draws"])
table

The arm carries **no p-value** `[§13]`: a second test of the primary quantity is a
second chance at it. What it carries instead is a second worst-residual-SMD, which is the number the
sensitivity exists to produce.

## 3. Write it

In [ ]:
path = M.save_table("Table_S1", table, (
    "Table S1. Primary 90-day mRS result in the [§7] overlap population, and the [§13] full-covariate "
    "propensity sensitivity beside it. The design rows are what makes the two comparable; the effect "
    "rows are the estimate.",
    *M.primary_clauses(bundle),
    *report.arm_clauses(bundle.arm),
    M.provenance_clause(bundle)))

print(path)
pd.read_csv(path, comment="#").tail(8)

## 4. Reading it

- **The six differences do not all point the same way.** The caption says so because the run says so,
  and `[§16a]` requires it: an odds ratio near 1 summarising differences that disagree in direction is
  not a null result.
- **The sensitivity moves the estimate very little and the imbalance quite a lot.** That is what it is
  for. It does not license the reading that unmeasured confounding is therefore small `[§13]`.
- **Both specifications ran on the same patients**, which the `differs only in specification` clause
  states as a computed fact rather than an assumption.